# Classificadores por grupo racial — Adult Income


**Fluxo:**
1. Carregar e preparar os dados
2. Split estratificado por raça
3. Treinar 8 modelos por grupo racial e avaliar só no teste da própria raça
4. Persistir modelos + tabela de teste única
5. Comparar métricas (acurácia, F1, ROC AUC)


## Modelos considerados

| modelo | Nome | Tipo |
|--------|------|------|
| LogisticRegression | Regressão Logística | Linear |
| SVC_Linear | SVC Linear | Linear |
| SGD_Logistic | Regressão Logística (SGD) | Linear |
| SVC_RBF | SVC Não Linear (RBF) | Não linear |
| GaussianNB | Naive Bayes Gaussiano | Não linear |
| RandomForest | Floresta Aleatória | Não linear |
| DecisionTree | Árvore de Decisão | Não linear |
| QDA | Análise Discriminante Quadrática | Não linear |

Cada modelo é treinado **separadamente** em cada grupo racial e avaliado **apenas** no teste da mesma raça.

## 1. Imports

In [31]:
from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

IRT_DIR = Path(r"C:\Users\User\Documents\Mestrado\UFPE\IRT")
TREINO_DIR = IRT_DIR / "treino"
TREINO_DIR.mkdir(exist_ok=True)

CACHE_TEST = TREINO_DIR / "test_df.parquet"
CACHE_RESULTS = TREINO_DIR / "results_df.parquet"
CACHE_SPLIT_META = TREINO_DIR / "split_meta.joblib"

FORCE_RETRAIN = True  # True para forçar retreino e sobrescrever


def slug_raca(raca: str) -> str:
    """Nome de pasta seguro a partir do valor de race."""
    return re.sub(r"[^\w.-]+", "_", str(raca).strip())


def pasta_raca(raca: str) -> Path:
    p = TREINO_DIR / slug_raca(raca)
    p.mkdir(parents=True, exist_ok=True)
    return p


def path_modelo(raca: str, modelo: str) -> Path:
    return pasta_raca(raca) / f"{modelo}.joblib"


## 2. Carregar dados

Dataset Adult (UCI). Alvo: `income` (>50K ou <=50K).

In [32]:
COLUNAS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

df = pd.read_csv(URL, names=COLUNAS, na_values="?", skipinitialspace=True)
df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Preparar dados

In [33]:
df.columns = [str(c).strip().lower().replace("-", "_") for c in df.columns]

target_col = next(c for c in ["income", "class", "target"] if c in df.columns)
df["_target"] = df[target_col].astype(str).str.contains(">50").astype(int)

print(f"Coluna alvo: {target_col}")
print(f"Valores únicos: {df[target_col].unique()}")


Coluna alvo: income
Valores únicos: <ArrowStringArray>
['<=50K', '>50K']
Length: 2, dtype: str


In [34]:
print("Distribuição do alvo (proporção):")
print(df["_target"].value_counts(normalize=True).round(3))

print("\nAmostras por raça:")
print(df["race"].value_counts())


Distribuição do alvo (proporção):
_target
0    0.759
1    0.241
Name: proportion, dtype: float64

Amostras por raça:
race
White                 27816
Black                  3124
Asian-Pac-Islander     1039
Amer-Indian-Eskimo      311
Other                   271
Name: count, dtype: int64


## 4. Funções auxiliares

In [35]:
def build_models():
    """Retorna os 8 classificadores (3 lineares + 5 não lineares)."""
    return {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "SVC_Linear": SVC(kernel="linear", probability=True, random_state=42),
        "SGD_Logistic": SGDClassifier(
            loss="log_loss", max_iter=1000, random_state=42
        ),
        "SVC_RBF": SVC(kernel="rbf", probability=True, random_state=42),
        "GaussianNB": GaussianNB(),
        "RandomForest": RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        # sklearn>=1.9: com n_amostras <= n_features por classe, svd falha;
        # eigen + shrinkage regulariza a covariância.
        "QDA": QuadraticDiscriminantAnalysis(
            solver="eigen", shrinkage=0.5
        ),
    }


In [36]:
def make_preprocessor(X):
    """Pré-processa numéricas (imputar + padronizar) e categóricas (imputar + one-hot)."""
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


In [37]:
def avaliar(pipe, X_te, y_te):
    """Calcula métricas de classificação no teste por raça."""
    pred = pipe.predict(X_te)
    try:
        proba = pipe.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, proba) if y_te.nunique() > 1 else np.nan
    except Exception:
        auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    return {
        "n_teste": len(y_te),
        "acuracia": accuracy_score(y_te, pred),
        "f1": f1_score(y_te, pred, zero_division=0),
        "roc_auc": auc,
        "erro": (fp + fn) / len(y_te),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
    }


## 5. Treinar e avaliar por raça

- Split único estratificado por `race` (`TEST_SIZE = 10%`)
- Avaliação **somente** no teste da própria raça (`tipo_teste = por_raca`)
- Cada pipeline vai para `treino/<raça>/<modelo>.joblib`
- Todos os dados de teste ficam em **uma** tabela: `treino/test_df.parquet`

In [38]:
feature_cols = [c for c in df.columns if c not in [target_col, "_target", "race"]]

RANDOM_STATE = 42
TEST_SIZE = 0.10
MIN_AMOSTRAS = 50
MODEL_NAMES = list(build_models().keys())


In [39]:
def cache_modelos_ok(racas):
    """True se todos os .joblib das raças/modelos existem."""
    return all(
        path_modelo(race, name).exists()
        for race in racas
        for name in MODEL_NAMES
    )


def carregar_pipes(racas):
    pipes = {}
    for race in racas:
        for name in MODEL_NAMES:
            pipes[(race, name)] = joblib.load(path_modelo(race, name))
    return pipes


def salvar_pipes(pipes):
    for (race, name), pipe in pipes.items():
        dest = path_modelo(race, name)
        joblib.dump(pipe, dest)
        print(f"  salvo: {dest.as_posix()}")


meta_ok = (
    not FORCE_RETRAIN
    and CACHE_TEST.exists()
    and CACHE_RESULTS.exists()
    and CACHE_SPLIT_META.exists()
)

if meta_ok:
    split_meta = joblib.load(CACHE_SPLIT_META)
    racas_validas = list(split_meta["racas_validas"])
    feature_cols = split_meta["feature_cols"]
else:
    racas_validas = None

cache_ok = meta_ok and racas_validas is not None and cache_modelos_ok(racas_validas)

if cache_ok:
    print(f"Carregando de {TREINO_DIR.resolve()} (sem retreinar)")
    test_df = pd.read_parquet(CACHE_TEST)
    results_df = pd.read_parquet(CACHE_RESULTS)
    pipes = carregar_pipes(racas_validas)

    splits = {}
    for race in racas_validas:
        te = test_df[test_df["race"] == race]
        splits[race] = {
            "X_te": te[feature_cols],
            "y_te": te["_target"],
        }

    print(
        f"OK — {len(pipes)} pipelines | teste: {len(test_df)} linhas "
        f"| raças: {racas_validas}"
    )
else:
    # --- 1. Split único estratificado por raça ---
    racas_validas = (
        df.groupby("race").size().loc[lambda s: s >= MIN_AMOSTRAS].index.tolist()
    )
    df_valid = df[df["race"].isin(racas_validas)].copy()

    idx_tr, idx_te = train_test_split(
        df_valid.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df_valid["race"],
    )

    train_df = df_valid.loc[idx_tr]
    test_df = df_valid.loc[idx_te].copy()

    # --- 2. Treinar por raça e avaliar só no teste da própria raça ---
    results = []
    pipes = {}
    splits = {}

    for race in racas_validas:
        tr = train_df[train_df["race"] == race]
        te = test_df[test_df["race"] == race]
        X_tr, y_tr = tr[feature_cols], tr["_target"]
        X_te, y_te = te[feature_cols], te["_target"]
        X_all = df_valid.loc[df_valid["race"] == race, feature_cols]

        splits[race] = {"X_te": X_te, "y_te": y_te}

        print(f"\n=== Treinando raça: {race} (n_treino={len(X_tr)}, n_teste={len(X_te)}) ===")
        for name, model in build_models().items():
            pipe = Pipeline([("pre", make_preprocessor(X_all)), ("clf", model)])
            pipe.fit(X_tr, y_tr)
            pipes[(race, name)] = pipe

            m = avaliar(pipe, X_te, y_te)
            results.append({
                "raca": race,
                "modelo": name,
                "tipo_teste": "por_raca",
                "n_treino": len(X_tr),
                **m,
            })

    results_df = pd.DataFrame(results).round(3)

    # --- 3. Persistir: modelos por pasta + tabela única de teste ---
    print(f"\nSalvando modelos em {TREINO_DIR.resolve()}")
    salvar_pipes(pipes)

    test_df.to_parquet(CACHE_TEST)
    results_df.to_parquet(CACHE_RESULTS)
    joblib.dump(
        {
            "racas_validas": racas_validas,
            "feature_cols": feature_cols,
            "random_state": RANDOM_STATE,
            "test_size": TEST_SIZE,
        },
        CACHE_SPLIT_META,
    )

    print(f"\nTabela de teste única: {CACHE_TEST} ({len(test_df)} linhas)")
    print(f"Métricas: {CACHE_RESULTS}")
    print(f"Meta: {CACHE_SPLIT_META}")

for race in sorted(results_df["raca"].unique()):
    print(f"\n=== {race} ===")
    display(results_df[results_df["raca"] == race])



=== Treinando raça: Amer-Indian-Eskimo (n_treino=280, n_teste=31) ===


c:\Users\User\Documents\Mestrado\UFPE\IRT\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\User\Documents\Mestrado\UFPE\IRT\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



=== Treinando raça: Asian-Pac-Islander (n_treino=935, n_teste=104) ===


c:\Users\User\Documents\Mestrado\UFPE\IRT\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\User\Documents\Mestrado\UFPE\IRT\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



=== Treinando raça: Black (n_treino=2811, n_teste=313) ===


...

,raca,modelo,tipo_teste,n_treino,n_teste,acuracia,f1,roc_auc,erro,fpr,fnr
16,Black,LogisticRegression,por_raca,2811,313,0.927,0.596,0.932,0.073,0.035,0.433
17,Black,SVC_Linear,por_raca,2811,313,0.920,0.528,0.932,0.080,0.032,0.533
18,Black,SGD_Logistic,por_raca,2811,313,0.927,0.511,0.924,0.073,0.018,0.600
19,Black,SVC_RBF,por_raca,2811,313,0.930,0.542,0.925,0.070,0.018,0.567
20,Black,GaussianNB,por_raca,2811,313,0.355,0.217,0.631,0.645,0.707,0.067
21,Black,RandomForest,por_raca,2811,313,0.930,0.593,0.938,0.070,0.028,0.467
22,Black,DecisionTree,por_raca,2811,313,0.895,0.535,0.778,0.105,0.078,0.367
23,Black,QDA,por_raca,2811,313,0.923,0.333,0.913,0.077,0.000,0.800



=== Other ===


,raca,modelo,tipo_teste,n_treino,n_teste,acuracia,f1,roc_auc,erro,fpr,fnr
24,Other,LogisticRegression,por_raca,244,27,0.889,0.400,0.931,0.111,0.042,0.667
25,Other,SVC_Linear,por_raca,244,27,0.889,0.400,0.944,0.111,0.042,0.667
26,Other,SGD_Logistic,por_raca,244,27,0.926,0.500,0.986,0.074,0.000,0.667
27,Other,SVC_RBF,por_raca,244,27,0.889,0.000,0.944,0.111,0.000,1.000
28,Other,GaussianNB,por_raca,244,27,0.667,0.182,0.521,0.333,0.292,0.667
29,Other,RandomForest,por_raca,244,27,0.926,0.500,0.938,0.074,0.000,0.667
30,Other,DecisionTree,por_raca,244,27,0.926,0.500,0.667,0.074,0.000,0.667
31,Other,QDA,por_raca,244,27,0.889,0.000,0.931,0.111,0.000,1.000



=== White ===


,raca,modelo,tipo_teste,n_treino,n_teste,acuracia,f1,roc_auc,erro,fpr,fnr
32,White,LogisticRegression,por_raca,25034,2782,0.845,0.685,0.903,0.155,0.083,0.357
33,White,SVC_Linear,por_raca,25034,2782,0.839,0.666,0.899,0.161,0.080,0.388
34,White,SGD_Logistic,por_raca,25034,2782,0.841,0.676,0.900,0.159,0.084,0.368
35,White,SVC_RBF,por_raca,25034,2782,0.850,0.682,0.899,0.150,0.067,0.384
36,White,GaussianNB,por_raca,25034,2782,0.515,0.510,0.716,0.485,0.644,0.037
37,White,RandomForest,por_raca,25034,2782,0.844,0.687,0.903,0.156,0.090,0.344
38,White,DecisionTree,por_raca,25034,2782,0.802,0.634,0.754,0.198,0.146,0.346
39,White,QDA,por_raca,25034,2782,0.792,0.415,0.889,0.208,0.027,0.719


### Filtrar o teste por raça (sem reprocessar)

A tabela `test_df` já contém todas as raças. Para análise de um grupo:

In [40]:
# Exemplo: carregar a tabela única e filtrar
test_unico = pd.read_parquet(CACHE_TEST)
print("Contagem por raça no teste:")
display(test_unico["race"].value_counts())

RACA_FILTRO = "Black"  # altere conforme necessário
teste_raca = test_unico[test_unico["race"] == RACA_FILTRO]
print(f"\nFiltro race == {RACA_FILTRO!r}: {len(teste_raca)} instâncias")
display(teste_raca.head())


Contagem por raça no teste:


race
White                 2782
Black                  313
Asian-Pac-Islander     104
Amer-Indian-Eskimo      31
Other                   27
Name: count, dtype: int64


Filtro race == 'Black': 313 instâncias


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,_target
2232,45,Private,339506,HS-grad,9,Never-married,Sales,Not-in-family,Black,Male,0,0,40,United-States,<=50K,0
28247,47,Private,264244,HS-grad,9,Married-spouse-absent,Craft-repair,Not-in-family,Black,Female,0,0,40,United-States,<=50K,0
15736,33,Federal-gov,615893,Masters,14,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40,Nicaragua,<=50K,0
12527,48,Private,167967,HS-grad,9,Separated,Machine-op-inspct,Unmarried,Black,Female,0,0,40,United-States,<=50K,0
124,19,Private,301606,Some-college,10,Never-married,Other-service,Own-child,Black,Male,0,0,35,United-States,<=50K,0


## 6. Análise dos resultados (teste por raça)

In [41]:
por_raca_df = results_df[results_df["tipo_teste"] == "por_raca"].copy()
racas_cols = sorted(por_raca_df["raca"].unique())


def montar_tabela(metric):
    """Modelos nas linhas, raças nas colunas."""
    tab = por_raca_df.pivot(index="modelo", columns="raca", values=metric).round(3)
    return tab.reindex(columns=racas_cols)


print("=== Acurácia ===")
display(montar_tabela("acuracia"))

print("=== F1 ===")
display(montar_tabela("f1"))

print("=== ROC AUC ===")
display(montar_tabela("roc_auc"))


=== Acurácia ===


raca,Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Other,White
modelo,,,,,
DecisionTree,0.677,0.769,0.895,0.926,0.802
GaussianNB,0.613,0.365,0.355,0.667,0.515
LogisticRegression,0.903,0.865,0.927,0.889,0.845
QDA,0.839,0.837,0.923,0.889,0.792
RandomForest,0.806,0.856,0.930,0.926,0.844
SGD_Logistic,0.935,0.769,0.927,0.926,0.841
SVC_Linear,0.903,0.837,0.920,0.889,0.839
SVC_RBF,0.806,0.865,0.930,0.889,0.850


=== F1 ===


raca,Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Other,White
modelo,,,,,
DecisionTree,0.286,0.636,0.535,0.500,0.634
GaussianNB,0.538,0.468,0.217,0.182,0.510
LogisticRegression,0.727,0.741,0.596,0.400,0.685
QDA,0.444,0.622,0.333,0.000,0.415
RandomForest,0.250,0.754,0.593,0.500,0.687
SGD_Logistic,0.833,0.500,0.511,0.500,0.676
SVC_Linear,0.727,0.679,0.528,0.400,0.666
SVC_RBF,0.250,0.759,0.542,0.000,0.682


=== ROC AUC ===


raca,Amer-Indian-Eskimo,Asian-Pac-Islander,Black,Other,White
modelo,,,,,
DecisionTree,0.539,0.755,0.778,0.667,0.754
GaussianNB,0.750,0.567,0.631,0.521,0.716
LogisticRegression,0.946,0.934,0.932,0.931,0.903
QDA,0.845,0.922,0.913,0.931,0.889
RandomForest,0.875,0.931,0.938,0.938,0.903
SGD_Logistic,0.929,0.842,0.924,0.986,0.900
SVC_Linear,0.958,0.922,0.932,0.944,0.899
SVC_RBF,0.881,0.949,0.925,0.944,0.899


In [42]:
print("=== Melhor modelo por raça — teste por_raca (ROC AUC) ===")
best = (
    por_raca_df.sort_values("roc_auc", ascending=False)
    .groupby("raca", observed=True)
    .first()[["modelo", "roc_auc", "acuracia", "f1", "n_teste"]]
)
display(best)


=== Melhor modelo por raça — teste por_raca (ROC AUC) ===


,modelo,roc_auc,acuracia,f1,n_teste
raca,,,,,
Amer-Indian-Eskimo,SVC_Linear,0.958,0.903,0.727,31
Asian-Pac-Islander,SVC_RBF,0.949,0.865,0.759,104
Black,RandomForest,0.938,0.930,0.593,313
Other,SGD_Logistic,0.986,0.926,0.500,27
White,LogisticRegression,0.903,0.845,0.685,2782
